# Draw a single ShearNet stamp pair

Calls ShearNet's **own** dataset generator to render **exactly one** galaxy, using the
same PSF file, detection catalog and random seed as the fiducial training run, and
saves the noisy ("dirty") galaxy stamp together with its PSF stamp to an `.npz`.

The `.npz` is the input for the architecture figure, so the stamps shown in the paper
figure are the literal stamps the network is trained on -- not an illustration.

**Why this is reproducible.** `generate_dataset` seeds every object independently, so
requesting `samples=1` returns byte-for-byte the same stamp as object `0` of the full
300k-object run. That is verified by an assertion at the bottom of this notebook.

**Environment.** This notebook needs ShearNet itself (and therefore GalSim / ngmix /
JAX), which the light plotting environment deliberately does not carry. Use the
`shearnet-plots-sim` environment -- see the repository README for the pinned ShearNet
commit and setup instructions.

## 1. Parameters

`CONFIG_PATH` points at a ShearNet config in a ShearNet checkout; every simulation
setting (PSF file, catalog, seed, stamp size, pixel scale, noise level) is read
**from that config**, so this notebook cannot silently drift from the real run.

`POPULATION` selects which catalog and seed to reproduce:

| value | catalog | seed | reproduces |
|---|---|---|---|
| `"train"` | `paths.train_catalog` | `train.seed` | the training population |
| `"eval"` | `paths.eval_catalog` | `eval.seed` | the held-out benchmark population |

In [1]:
from pathlib import Path

# --- point these at your ShearNet checkout -------------------------------------
SHEARNET_REPO = Path("~/ShearNet").expanduser()
CONFIG_PATH   = SHEARNET_REPO / "research/unit_tests/fourth/config.yaml"

# "train" -> train_catalog + train.seed ; "eval" -> eval_catalog + eval.seed
POPULATION = "train"

# Which object of the population to draw. 0 (the default) issues a literal
# samples=1 call. A nonzero index renders index+1 objects and keeps the last,
# which is the only way to reach object k while preserving its exact per-object
# seed -- use it if object 0 happens to be an unphotogenic galaxy.
GALAXY_INDEX = 0

OUT_NPZ = Path("stamps") / "single_stamp.npz"
PREVIEW = True

## 2. Imports and provenance

Record the exact ShearNet commit so the `.npz` can always be traced back to the code that made it.

In [2]:
import subprocess
import warnings

import numpy as np
import matplotlib.pyplot as plt

import shearnet
from shearnet.config.config_handler import Config
from shearnet.core.dataset import split_combined_images
from shearnet.core.specs import DatasetSpec


def git_commit(repo: Path) -> str:
    """Full commit hash of a checkout, or a marker if it cannot be resolved."""
    try:
        out = subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "HEAD"],
            capture_output=True, text=True, check=True,
        )
        dirty = subprocess.run(
            ["git", "-C", str(repo), "status", "--porcelain"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        return out.stdout.strip() + ("-dirty" if dirty else "")
    except Exception:
        return "unknown"


SHEARNET_COMMIT = git_commit(SHEARNET_REPO)
print("shearnet package :", Path(shearnet.__file__).parent)
print("shearnet commit  :", SHEARNET_COMMIT)
print("config           :", CONFIG_PATH)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


shearnet package : /home/adfield/ShearNet/shearnet
shearnet commit  : unknown
config           : /home/adfield/ShearNet/research/unit_test_variations/fourth_inloop_shearnet/config.yaml


## 3. Build the dataset spec from the config

`DatasetSpec.from_config` is the same object the training CLI builds, so whatever it
resolves here is what a real run would use. We override **only** `samples` (and the
catalog/seed when reproducing the eval population).

In [3]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Config not found: {CONFIG_PATH}\n"
        "Set SHEARNET_REPO / CONFIG_PATH in the parameters cell to your checkout."
    )

config = Config(str(CONFIG_PATH))
spec = DatasetSpec.from_config(config)

# from_config always takes the *training* catalog and seed; switch when asked.
if POPULATION == "eval":
    spec.cosmos_cat_fname = config.get("paths.eval_catalog")
    spec.seed = config.get("eval.seed")
elif POPULATION != "train":
    raise ValueError(f"POPULATION must be 'train' or 'eval', got {POPULATION!r}")

spec.samples = GALAXY_INDEX + 1     # 1 for the default index 0
spec.return_psf = True              # we need the PSF stamp alongside the galaxy
spec.nproc = 1                      # serial: identical output, no pool overhead

print(f"population    : {POPULATION}")
print(f"seed          : {spec.seed}")
print(f"exp (PSF mode): {spec.exp}")
print(f"psf file/dir  : {spec.psf_file_or_dir}")
print(f"catalog       : {spec.cosmos_cat_fname}")
print(f"stamp         : {spec.npix} px @ {spec.scale} arcsec/px")
print(f"noise sd      : {spec.nse_sd}")
print(f"hlr / flux    : {spec.hlr_type} / {spec.flux_type}")
print(f"samples       : {spec.samples}")

population    : train
seed          : 42
exp (PSF mode): superbit
psf file/dir  : /home/adfield/weak_lensing/superbit-lensing-jax-test/simulated_data/sim_utils/emp_psfs/emp_psfs_best/psfex_all_except_best
catalog       : /home/adfield/ShearNet/cosmos_catalog_train.fits
stamp         : 53 px @ 0.141 arcsec/px
noise sd      : 12.719674
hlr / flux    : catalog / catalog
samples       : 1


### Check the inputs actually exist

The fiducial config carries absolute cluster paths. If either the PSF model or the
detection catalog is missing, ShearNet silently falls back to a **synthetic random
catalog** -- which would quietly produce a galaxy that is not from the real
population. Fail loudly instead.

In [4]:
psf_path = Path(spec.psf_file_or_dir) if spec.psf_file_or_dir else None
cat_path = Path(spec.cosmos_cat_fname) if spec.cosmos_cat_fname else None

problems = []
if psf_path is None or not psf_path.exists():
    problems.append(f"PSF model not found: {spec.psf_file_or_dir}")
if cat_path is None or not cat_path.exists():
    problems.append(
        f"Detection catalog not found: {spec.cosmos_cat_fname}\n"
        "    ShearNet would fall back to a SYNTHETIC random catalog, so the drawn "
        "galaxy would NOT come from the real detection population."
    )

if problems:
    raise FileNotFoundError(
        "Cannot reproduce the fiducial population:\n  - "
        + "\n  - ".join(problems)
        + "\n\nEither run this notebook where those paths resolve, or edit the "
          "config / spec fields above to point at your local copies."
    )

print("PSF model        :", psf_path)
print("detection catalog:", cat_path)

PSF model        : /home/adfield/weak_lensing/superbit-lensing-jax-test/simulated_data/sim_utils/emp_psfs/emp_psfs_best/psfex_all_except_best
detection catalog: /home/adfield/ShearNet/cosmos_catalog_train.fits


## 4. Draw the galaxy

`spec.build()` dispatches to `shearnet.core.dataset.generate_dataset`.

In [5]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    images, labels = spec.build()

galaxy_all, psf_all = split_combined_images(images, has_psf=True)

galaxy = np.asarray(galaxy_all[GALAXY_INDEX], dtype=np.float64)
psf    = np.asarray(psf_all[GALAXY_INDEX],    dtype=np.float64)
label  = np.asarray(labels[GALAXY_INDEX],     dtype=np.float64)
output_keys = tuple(spec.output_keys)

print("galaxy stamp :", galaxy.shape, f"min={galaxy.min():.3f} max={galaxy.max():.3f}")
print("psf stamp    :", psf.shape,    f"min={psf.min():.6f} max={psf.max():.6f}")
print("labels       :", dict(zip(output_keys, label)))

ValueError: build() materialises a dataset, which dataset.generation: inloop deliberately never does -- use build_inloop_generator().

## 5. Preview

In [ ]:
from matplotlib.colors import LogNorm

if PREVIEW:
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    # Galaxy: ordinary linear scale
    im_gal = axes[0].imshow(
        galaxy,
        origin="lower",
        cmap="viridis",
    )
    axes[0].set_title("galaxy (PSF-convolved + noise)", fontsize=9)
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    fig.colorbar(im_gal, ax=axes[0], fraction=0.046)

    # PSF: logarithmic scale
    positive_psf = psf[psf > 0]

    im_psf = axes[1].imshow(
        psf,
        origin="lower",
        cmap="viridis",
        norm=LogNorm(
            vmin=positive_psf.min(),
            vmax=psf.max(),
        ),
    )
    axes[1].set_title("PSF model", fontsize=9)
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    fig.colorbar(im_psf, ax=axes[1], fraction=0.046)

    fig.tight_layout()
    plt.show()

## 6. Save

Everything needed to regenerate this exact pair travels with the arrays, so the
figure code never has to guess where the stamps came from.

In [ ]:
OUT_NPZ.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_NPZ,
    galaxy=galaxy,
    psf=psf,
    labels=label,
    output_keys=np.array(output_keys),
    # --- provenance -----------------------------------------------------------
    shearnet_commit=SHEARNET_COMMIT,
    config_path=str(CONFIG_PATH),
    population=POPULATION,
    galaxy_index=GALAXY_INDEX,
    seed=spec.seed,
    exp=spec.exp,
    psf_file_or_dir=str(spec.psf_file_or_dir),
    cosmos_cat_fname=str(spec.cosmos_cat_fname),
    npix=spec.npix,
    scale=spec.scale,
    nse_sd=spec.nse_sd,
    psf_fwhm=spec.psf_fwhm,
    hlr_type=spec.hlr_type,
    flux_type=spec.flux_type,
)

print(f"wrote {OUT_NPZ.resolve()}  ({OUT_NPZ.stat().st_size / 1024:.1f} KiB)")

## 7. Reproducibility check

Re-render with a larger `samples` and confirm object `GALAXY_INDEX` is unchanged. This
is what licenses drawing one galaxy instead of the full population: per-object seeding
means a `samples=1` call is not an approximation of the real run, it *is* the real run's
first object.

In [ ]:
import dataclasses

check_spec = dataclasses.replace(spec, samples=GALAXY_INDEX + 4)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    check_images, check_labels = check_spec.build()

check_gal, check_psf = split_combined_images(check_images, has_psf=True)

assert np.array_equal(galaxy, check_gal[GALAXY_INDEX]), "galaxy stamp is not reproducible"
assert np.array_equal(psf,    check_psf[GALAXY_INDEX]),  "psf stamp is not reproducible"
assert np.array_equal(label,  check_labels[GALAXY_INDEX]), "labels are not reproducible"
print(f"OK: object {GALAXY_INDEX} is identical when rendered as part of a "
      f"{check_spec.samples}-object population.")